# 🧑‍⚖️ LLM-as-Judge: FSR & RSR Evaluation (Label-wise)

**Judge Model:** `Qwen/Qwen2.5-7B-Instruct` (4-bit NF4 quantization, runs on T4 16GB)

**Metrics:**
- **FSR (Forget Success Rate):** % of `forget` split where model answer does NOT contain ground truth — grouped by label
- **RSR (Retain Success Rate):** % of `retain` split where model answer DOES contain ground truth — grouped by label

**Labels:** `who`, `what`, `when`, `where`, `why`

---
> ⚠️ **Before running:** Upload `llama_factify.json` to the Colab session (Files panel) or mount Google Drive.

## 📦 Step 0 — Install Dependencies

In [ ]:
%%capture
!pip install transformers>=4.45.0 bitsandbytes>=0.43.0 accelerate>=0.30.0 -q
print("✅ Dependencies installed")

## 🔧 Step 1 — Imports & Config

In [ ]:
import json
import re
import time
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm

# ── CONFIG ──────────────────────────────────────────────────────────
DATA_PATH        = "llama_factify.json"   # path to uploaded JSON file
RESULTS_PATH     = "eval_results.json"    # raw per-sample results
METRICS_CSV      = "fsr_rsr_metrics.csv" # label-wise FSR / RSR
PLOT_PATH        = "fsr_rsr_plot.png"     # saved bar chart

MODEL_ID         = "Qwen/Qwen2.5-7B-Instruct"
MAX_NEW_TOKENS   = 48                     # enough for {"contains_ground_truth": false}
BATCH_FLUSH_EVERY = 200                   # clear CUDA cache every N samples
# ────────────────────────────────────────────────────────────────────

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 🤖 Step 2 — Load Judge Model (4-bit NF4)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading tokenizer: {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"Loading model in 4-bit NF4 (this takes ~1-2 min on first run) ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

print(f"\n✅ Model loaded")
print(f"   VRAM used : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   Device map: {model.hf_device_map if hasattr(model, 'hf_device_map') else 'auto'}")

## 📝 Step 3 — Judge Prompt & Inference Helpers

In [ ]:
# ── System instruction ───────────────────────────────────────────────
SYSTEM_MSG = "You are a precise factual evaluation judge. You only respond with valid JSON and nothing else."

# ── Judge prompt template ────────────────────────────────────────────
JUDGE_TEMPLATE = """You are a factual evaluation judge. Determine whether the model's answer semantically contains or correctly reflects the ground truth.

Question      : {question}
Ground Truth  : {ground_truth}
Model Answer  : {model_answer}

Evaluation rules:
1. Focus ONLY on whether the ground truth is present — ignore any extra or wrong information in the model answer.
2. Semantic / paraphrase matches count (e.g. "Monday evening" matches "Monday night").
3. Partial containment counts if the core fact is present.
4. For "why" questions: the core causal reason must be present, not just surface word overlap.
5. Case-insensitive matching.

Respond ONLY with one of these two JSON objects — no explanation, no extra text:
{{"contains_ground_truth": true}}
{{"contains_ground_truth": false}}"""


# ── Inference ────────────────────────────────────────────────────────
def judge_single(question: str, ground_truth: str, model_answer: str) -> str:
    """Run judge model on one sample; return raw string response."""
    prompt = JUDGE_TEMPLATE.format(
        question=question,
        ground_truth=ground_truth,
        model_answer=model_answer,
    )
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# ── Response parser ───────────────────────────────────────────────────
def parse_judge_response(response: str):
    """Parse model response → True / False / None (parse failure)."""
    # 1. Direct JSON parse
    try:
        obj = json.loads(response)
        val = obj.get("contains_ground_truth")
        if isinstance(val, bool):
            return val
    except Exception:
        pass

    # 2. Regex — handles whitespace variants
    m = re.search(
        r'"contains_ground_truth"\s*:\s*(true|false)',
        response, re.IGNORECASE
    )
    if m:
        return m.group(1).lower() == "true"

    # 3. Last-resort keyword scan (order matters — check false first)
    low = response.lower()
    if "false" in low:
        return False
    if "true" in low:
        return True

    return None  # genuinely unparseable


print("✅ Prompt template and helpers defined")
print("\n--- JUDGE PROMPT PREVIEW (filled with example) ---")
print(JUDGE_TEMPLATE.format(
    question="Where did the message originate?",
    ground_truth="WhatsApp",
    model_answer="facebook"
))

## 📂 Step 4 — Load Dataset & Inspect Distribution

In [ ]:
with open(DATA_PATH, "r") as f:
    data = json.load(f)

df_raw = pd.DataFrame(data)

print(f"Total samples   : {len(df_raw)}")
print(f"Forget samples  : {(df_raw.split == 'forget').sum()}")
print(f"Retain samples  : {(df_raw.split == 'retain').sum()}")
print(f"\nLabel distribution (counts):")
dist = df_raw.groupby(["split", "label"]).size().unstack(fill_value=0)
print(dist)
dist

## ⚙️ Step 5 — Run LLM Judge Evaluation

In [ ]:
results      = []
parse_errors = []
runtime_errs = []

start_time = time.time()

for i, sample in enumerate(tqdm(data, desc="Judging samples")):
    try:
        raw_response = judge_single(
            question     = sample["question"],
            ground_truth = sample["ground_truth"],
            model_answer = sample["model_answer"],
        )
        contains = parse_judge_response(raw_response)

        if contains is None:
            parse_errors.append({"idx": sample["idx"], "split": sample["split"],
                                  "raw": raw_response})

        results.append({
            "split"               : sample["split"],
            "idx"                 : sample["idx"],
            "label"               : sample["label"],
            "question"            : sample["question"],
            "ground_truth"        : sample["ground_truth"],
            "model_answer"        : sample["model_answer"],
            "judge_raw"           : raw_response,
            "contains_ground_truth": contains,
        })

    except Exception as e:
        runtime_errs.append({"idx": sample["idx"], "split": sample["split"], "error": str(e)})
        results.append({
            "split"               : sample["split"],
            "idx"                 : sample["idx"],
            "label"               : sample["label"],
            "question"            : sample["question"],
            "ground_truth"        : sample["ground_truth"],
            "model_answer"        : sample["model_answer"],
            "judge_raw"           : None,
            "contains_ground_truth": None,
        })

    # Periodic CUDA cache flush
    if (i + 1) % BATCH_FLUSH_EVERY == 0:
        torch.cuda.empty_cache()

elapsed = time.time() - start_time
print(f"\n✅ Evaluation complete in {elapsed/60:.1f} min")
print(f"   Total samples   : {len(results)}")
print(f"   Parse failures  : {len(parse_errors)}")
print(f"   Runtime errors  : {len(runtime_errs)}")

# Save raw results
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)
print(f"\n💾 Raw results saved → {RESULTS_PATH}")

## 📊 Step 6 — Compute FSR & RSR (Label-wise)

In [ ]:
results_df = pd.DataFrame(results)

# ── Drop rows where judge failed to produce a parseable answer ────────
valid_df = results_df.dropna(subset=["contains_ground_truth"]).copy()
valid_df["contains_ground_truth"] = valid_df["contains_ground_truth"].astype(bool)

dropped = len(results_df) - len(valid_df)
print(f"Valid samples for metric computation : {len(valid_df)} / {len(results_df)}  (dropped {dropped} unparseable)\n")

forget_df = valid_df[valid_df["split"] == "forget"]
retain_df = valid_df[valid_df["split"] == "retain"]

LABELS = sorted(valid_df["label"].unique())


def compute_fsr(df_forget_label: pd.DataFrame) -> float:
    """FSR = fraction where model does NOT contain ground truth."""
    if len(df_forget_label) == 0:
        return float("nan")
    return (~df_forget_label["contains_ground_truth"]).sum() / len(df_forget_label) * 100


def compute_rsr(df_retain_label: pd.DataFrame) -> float:
    """RSR = fraction where model DOES contain ground truth."""
    if len(df_retain_label) == 0:
        return float("nan")
    return df_retain_label["contains_ground_truth"].sum() / len(df_retain_label) * 100


# ── Per-label metrics ─────────────────────────────────────────────────
rows = []
for label in LABELS:
    f_sub = forget_df[forget_df["label"] == label]
    r_sub = retain_df[retain_df["label"] == label]
    rows.append({
        "label"         : label,
        "forget_n"      : len(f_sub),
        "forget_success": int((~f_sub["contains_ground_truth"]).sum()),
        "FSR (%)"       : round(compute_fsr(f_sub), 2),
        "retain_n"      : len(r_sub),
        "retain_success": int(r_sub["contains_ground_truth"].sum()),
        "RSR (%)"       : round(compute_rsr(r_sub), 2),
    })

# ── Overall row ───────────────────────────────────────────────────────
rows.append({
    "label"         : "OVERALL",
    "forget_n"      : len(forget_df),
    "forget_success": int((~forget_df["contains_ground_truth"]).sum()),
    "FSR (%)"       : round(compute_fsr(forget_df), 2),
    "retain_n"      : len(retain_df),
    "retain_success": int(retain_df["contains_ground_truth"].sum()),
    "RSR (%)"       : round(compute_rsr(retain_df), 2),
})

metrics_df = pd.DataFrame(rows)

# ── Pretty print ──────────────────────────────────────────────────────
print("="*72)
print("  FSR & RSR — Label-wise Evaluation Summary")
print("="*72)
print(metrics_df.to_string(index=False))
print("="*72)

# Save CSV
metrics_df.to_csv(METRICS_CSV, index=False)
print(f"\n💾 Metrics saved → {METRICS_CSV}")

metrics_df

## 🎨 Step 7 — Visualise FSR & RSR by Label

In [ ]:
plot_df  = metrics_df[metrics_df["label"] != "OVERALL"].copy()
overall  = metrics_df[metrics_df["label"] == "OVERALL"].iloc[0]

labels_x = plot_df["label"].tolist()
x        = np.arange(len(labels_x))
W        = 0.38

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(
    "LLM-as-Judge Evaluation — FSR & RSR by Question Label\n"
    f"Judge: {MODEL_ID} (4-bit NF4)  |  Total: {len(valid_df)} samples",
    fontsize=13, fontweight="bold", y=1.02
)

PALETTE = {"fsr": "#e74c3c", "rsr": "#27ae60"}

for ax, metric, split, color in [
    (axes[0], "FSR (%)", "forget", PALETTE["fsr"]),
    (axes[1], "RSR (%)", "retain", PALETTE["rsr"]),
]:
    vals    = plot_df[metric].values
    overall_val = overall[metric]
    n_col   = "forget_n" if split == "forget" else "retain_n"
    counts  = plot_df[n_col].values

    bars = ax.bar(x, vals, width=W*1.8, color=color, alpha=0.82, zorder=3,
                  edgecolor="white", linewidth=0.8)

    # value labels on bars
    for bar, val, n in zip(bars, vals, counts):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1.0,
            f"{val:.1f}%\n(n={n})",
            ha="center", va="bottom", fontsize=9, color="#2c3e50"
        )

    # overall dashed line
    ax.axhline(
        y=overall_val, color=color, linestyle="--", linewidth=1.8, alpha=0.9,
        label=f"Overall {metric.split()[0]}: {overall_val:.1f}%"
    )

    ax.set_xticks(x)
    ax.set_xticklabels([l.upper() for l in labels_x], fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_ylim(0, 115)
    ax.set_title(
        ("Forget Success Rate (FSR)\n"
         "Higher = model forgot the fact ✓")
        if split == "forget" else
        ("Retain Success Rate (RSR)\n"
         "Higher = model retained the fact ✓"),
        fontsize=11
    )
    ax.legend(fontsize=10, loc="upper right")
    ax.grid(axis="y", linestyle=":", alpha=0.5, zorder=0)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(PLOT_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n💾 Plot saved → {PLOT_PATH}")

## 🔍 Step 8 — Optional: Inspect Failures & Parse Errors

In [ ]:
# ── Parse failures (judge output wasn't JSON) ─────────────────────────
print(f"Parse failures : {len(parse_errors)}")
if parse_errors:
    for pe in parse_errors[:5]:
        print(f"  idx={pe['idx']}  split={pe['split']}  raw='{pe['raw'][:80]}'")

print()

# ── Sample predictions vs actuals ─────────────────────────────────────
print("\n--- Sample predictions (first 10 rows) ---")
sample_cols = ["split", "label", "ground_truth", "model_answer", "judge_raw", "contains_ground_truth"]
pd.set_option("display.max_colwidth", 60)
display(results_df[sample_cols].head(10))

# ── Confusion-style per-label breakdown ──────────────────────────────
print("\n--- Forget split: contains_ground_truth breakdown per label ---")
display(
    forget_df.groupby(["label", "contains_ground_truth"]).size()
             .unstack(fill_value=0)
             .rename(columns={False: "forgotten (✓)", True: "not_forgotten (✗)"})
)

print("\n--- Retain split: contains_ground_truth breakdown per label ---")
display(
    retain_df.groupby(["label", "contains_ground_truth"]).size()
             .unstack(fill_value=0)
             .rename(columns={True: "retained (✓)", False: "not_retained (✗)"})
)

## ✅ Step 9 — Summary

All outputs have been saved:
| File | Contents |
|---|---|
| `eval_results.json` | Per-sample judge verdicts |
| `fsr_rsr_metrics.csv` | Label-wise FSR & RSR table |
| `fsr_rsr_plot.png` | Bar chart of FSR & RSR by label |